In [ ]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from ipywidgets import interact, widgets

In [48]:
import pandas as pd
import psycopg2
from psycopg2 import OperationalError
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)
class PostgresConnection:
    def __init__(self, host, database, port, user, password):
        self.host = host
        self.database = database
        self.port = port
        self.user = user
        self.password = password
        self.connection = None
        self.cursor = None
    
    def connect(self):
        """Устанавливает соединение с БД и выводит версию PostgreSQL"""
        try:
            self.connection = psycopg2.connect(
                host=self.host,
                database=self.database,
                port=self.port,
                user=self.user,
                password=self.password
            )
            self.cursor = self.connection.cursor()
            
            # Получаем и выводим версию PostgreSQL
            self.cursor.execute("SELECT version();")
            version = self.cursor.fetchone()
            print(f"Успешное подключение к PostgreSQL. Версия: {version[0]}")
            
        except OperationalError as e:
            print(f"Ошибка подключения: {e}")
    
    def execute(self, query):
        """
        Выполняет SQL-запрос и возвращает результат в формате pandas DataFrame
        или скалярное значение для простых запросов
        """
        if not self.connection:
            print("Соединение не установлено. Сначала вызовите метод connect()")
            return None
            
        try:
            self.cursor.execute(query)
            
            # Для SELECT запросов возвращаем DataFrame
            if query.strip().upper().startswith('SELECT'):
                columns = [desc[0] for desc in self.cursor.description]
                data = self.cursor.fetchall()
                
                # Если результат содержит одну строку с одним значением
                if len(data) == 1 and len(data[0]) == 1:
                    return data[0][0]
                
                return pd.DataFrame(data, columns=columns)
            
            # Для других запросов (INSERT, UPDATE и т.д.) выполняем commit
            self.connection.commit()
            return f"Запрос выполнен успешно. Затронуто строк: {self.cursor.rowcount}"
            
        except Exception as e:
            self.connection.rollback()
            print(f"Ошибка выполнения запроса: {e}")
            return None
    
    def close(self):
        """Закрывает соединение с БД"""
        if self.cursor:
            self.cursor.close()
        if self.connection:
            self.connection.close()
            print("Соединение с базой данных закрыто")
    
    def __enter__(self):
        """Поддержка контекстного менеджера (with statement)"""
        self.connect()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Автоматическое закрытие соединения при выходе из контекста"""
        self.close()

In [49]:
master_conn = PostgresConnection(
    host="localhost",
    database="app_db",
    port=5432,
    user="pgadmin",
    password="adminpassword"
)
master_conn.connect()

replica_conn = PostgresConnection(
    host="localhost",
    database="app_db",
    port=5433,
    user="pgadmin",
    password="adminpassword"
)
replica_conn.connect()

Успешное подключение к PostgreSQL. Версия: PostgreSQL 15.13 on x86_64-pc-linux-musl, compiled by gcc (Alpine 14.2.0) 14.2.0, 64-bit
Успешное подключение к PostgreSQL. Версия: PostgreSQL 15.13 on x86_64-pc-linux-musl, compiled by gcc (Alpine 14.2.0) 14.2.0, 64-bit


In [50]:
# Получение списка таблиц на мастере 
result = master_conn.execute("select * from information_schema.tables where table_schema = 'public'")
result

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,app_db,public,categories,BASE TABLE,None,None,None,None,None,YES,NO,None
1,app_db,public,products,BASE TABLE,None,None,None,None,None,YES,NO,None
2,app_db,public,product_categories,BASE TABLE,None,None,None,None,None,YES,NO,None
3,app_db,public,users,BASE TABLE,None,None,None,None,None,YES,NO,None
4,app_db,public,orders,BASE TABLE,None,None,None,None,None,YES,NO,None
5,app_db,public,order_items,BASE TABLE,None,None,None,None,None,YES,NO,None
6,app_db,public,reviews,BASE TABLE,None,None,None,None,None,YES,NO,None


In [51]:
# Получение внешних ключей
result = master_conn.execute(""" 
SELECT conrelid::regclass AS table_name, 
       conname AS foreign_key, 
       pg_get_constraintdef(oid) 
FROM   pg_constraint 
WHERE  contype = 'f' 
AND    connamespace = 'public'::regnamespace   
ORDER  BY conrelid::regclass::text, contype DESC;""")
result

,table_name,foreign_key,pg_get_constraintdef
0,categories,fk_parent_category,FOREIGN KEY (parent_category_id) REFERENCES categories(category_id) ON DELETE SET NULL
1,order_items,fk_order,FOREIGN KEY (order_id) REFERENCES orders(order_id) ON DELETE CASCADE
2,order_items,fk_product,FOREIGN KEY (product_id) REFERENCES products(product_id) ON DELETE RESTRICT
3,orders,fk_user,FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE RESTRICT
4,product_categories,fk_product,FOREIGN KEY (product_id) REFERENCES products(product_id) ON DELETE CASCADE
5,product_categories,fk_category,FOREIGN KEY (category_id) REFERENCES categories(category_id) ON DELETE CASCADE
6,reviews,fk_user,FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
7,reviews,fk_product,FOREIGN KEY (product_id) REFERENCES products(product_id) ON DELETE CASCADE


In [55]:
# Получение списка атрибутов с ограничением уникальности
result = master_conn.execute(""" 
SELECT conrelid::regclass AS table_name, 
       conname AS foreign_key, 
       contype,
       pg_get_constraintdef(oid) 
FROM   pg_constraint 
WHERE contype = 'u' 
AND  connamespace = 'public'::regnamespace   
ORDER  BY conrelid::regclass::text, contype DESC;""")
result

,table_name,foreign_key,contype,pg_get_constraintdef
0,categories,categories_name_key,u,UNIQUE (name)
1,order_items,order_items_order_id_product_id_key,u,"UNIQUE (order_id, product_id)"
2,product_categories,product_categories_product_id_category_id_key,u,"UNIQUE (product_id, category_id)"
3,users,users_email_key,u,UNIQUE (email)


In [56]:
# Получение списка проверок CHECK
result = master_conn.execute(""" 
SELECT conrelid::regclass AS table_name, 
       conname AS foreign_key, 
       contype,
       pg_get_constraintdef(oid) 
FROM   pg_constraint 
WHERE  contype = 'c' 
AND   connamespace = 'public'::regnamespace   
ORDER  BY conrelid::regclass::text, contype DESC;""")
result

,table_name,foreign_key,contype,pg_get_constraintdef
0,order_items,order_items_quantity_check,c,CHECK ((quantity > 0))
1,orders,orders_status_check,c,"CHECK (((status)::text = ANY ((ARRAY['pending'::character varying, 'processing'::character varying, 'shipped'::character varying, 'delivered'::character varying, 'cancelled'::character varying])::text[])))"
2,orders,orders_total_check,c,CHECK ((total >= (0)::numeric))
3,products,products_price_check,c,CHECK ((price >= (0)::numeric))
4,products,products_stock_check,c,CHECK ((stock >= 0))
5,reviews,reviews_rating_check,c,CHECK (((rating >= 1) AND (rating <= 5)))
6,users,users_role_check,c,"CHECK (((role)::text = ANY ((ARRAY['customer'::character varying, 'admin'::character varying])::text[])))"


In [58]:
# Получение списка индексов
result = master_conn.execute(""" 
SELECT
    tablename,
    indexname,
    indexdef
FROM
    pg_indexes
WHERE
    schemaname = 'public'
ORDER BY
    tablename,
    indexname;""")
result

,tablename,indexname,indexdef
0,categories,categories_name_key,CREATE UNIQUE INDEX categories_name_key ON public.categories USING btree (name)
1,categories,categories_pkey,CREATE UNIQUE INDEX categories_pkey ON public.categories USING btree (category_id)
2,order_items,order_items_order_id_product_id_key,"CREATE UNIQUE INDEX order_items_order_id_product_id_key ON public.order_items USING btree (order_id, product_id)"
3,order_items,order_items_pkey,CREATE UNIQUE INDEX order_items_pkey ON public.order_items USING btree (order_item_id)
4,orders,orders_pkey,CREATE UNIQUE INDEX orders_pkey ON public.orders USING btree (order_id)
5,product_categories,product_categories_pkey,CREATE UNIQUE INDEX product_categories_pkey ON public.product_categories USING btree (product_category_id)
6,product_categories,product_categories_product_id_category_id_key,"CREATE UNIQUE INDEX product_categories_product_id_category_id_key ON public.product_categories USING btree (product_id, category_id)"
7,products,products_pkey,CREATE UNIQUE INDEX products_pkey ON public.products USING btree (product_id)
8,reviews,reviews_pkey,CREATE UNIQUE INDEX reviews_pkey ON public.reviews USING btree (review_id)
9,users,users_email_key,CREATE UNIQUE INDEX users_email_key ON public.users USING btree (email)


In [31]:
# Получение списка таблиц на реплике 
result = replica_conn.execute("select * from information_schema.tables where table_schema = 'public'")
result

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,app_db,public,categories,BASE TABLE,None,None,None,None,None,YES,NO,None
1,app_db,public,products,BASE TABLE,None,None,None,None,None,YES,NO,None
2,app_db,public,product_categories,BASE TABLE,None,None,None,None,None,YES,NO,None
3,app_db,public,users,BASE TABLE,None,None,None,None,None,YES,NO,None
4,app_db,public,orders,BASE TABLE,None,None,None,None,None,YES,NO,None
5,app_db,public,order_items,BASE TABLE,None,None,None,None,None,YES,NO,None
6,app_db,public,reviews,BASE TABLE,None,None,None,None,None,YES,NO,None


In [40]:
# Получение списка процедур на мастере 
result = master_conn.execute("""
select * 
from information_schema.routines
where routine_schema = 'public'
""")
result

,specific_catalog,specific_schema,specific_name,routine_catalog,routine_schema,routine_name,routine_type,module_catalog,module_schema,module_name,...,result_cast_interval_type,result_cast_interval_precision,result_cast_type_udt_catalog,result_cast_type_udt_schema,result_cast_type_udt_name,result_cast_scope_catalog,result_cast_scope_schema,result_cast_scope_name,result_cast_maximum_cardinality,result_cast_dtd_identifier
0,app_db,public,uuid_nil_16387,app_db,public,uuid_nil,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,app_db,public,uuid_ns_dns_16388,app_db,public,uuid_ns_dns,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,app_db,public,uuid_ns_url_16389,app_db,public,uuid_ns_url,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,app_db,public,uuid_ns_oid_16390,app_db,public,uuid_ns_oid,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,app_db,public,uuid_ns_x500_16391,app_db,public,uuid_ns_x500,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,app_db,public,uuid_generate_v1_16392,app_db,public,uuid_generate_v1,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,app_db,public,uuid_generate_v1mc_16393,app_db,public,uuid_generate_v1mc,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,app_db,public,uuid_generate_v3_16394,app_db,public,uuid_generate_v3,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
8,app_db,public,uuid_generate_v4_16395,app_db,public,uuid_generate_v4,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,app_db,public,uuid_generate_v5_16396,app_db,public,uuid_generate_v5,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [41]:
# Получение списка процедур на реплике 
result = replica_conn.execute("""
select * 
from information_schema.routines
where routine_schema = 'public'
""")
result

,specific_catalog,specific_schema,specific_name,routine_catalog,routine_schema,routine_name,routine_type,module_catalog,module_schema,module_name,...,result_cast_interval_type,result_cast_interval_precision,result_cast_type_udt_catalog,result_cast_type_udt_schema,result_cast_type_udt_name,result_cast_scope_catalog,result_cast_scope_schema,result_cast_scope_name,result_cast_maximum_cardinality,result_cast_dtd_identifier
0,app_db,public,uuid_nil_16387,app_db,public,uuid_nil,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,app_db,public,uuid_ns_dns_16388,app_db,public,uuid_ns_dns,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,app_db,public,uuid_ns_url_16389,app_db,public,uuid_ns_url,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,app_db,public,uuid_ns_oid_16390,app_db,public,uuid_ns_oid,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,app_db,public,uuid_ns_x500_16391,app_db,public,uuid_ns_x500,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,app_db,public,uuid_generate_v1_16392,app_db,public,uuid_generate_v1,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,app_db,public,uuid_generate_v1mc_16393,app_db,public,uuid_generate_v1mc,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,app_db,public,uuid_generate_v3_16394,app_db,public,uuid_generate_v3,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
8,app_db,public,uuid_generate_v4_16395,app_db,public,uuid_generate_v4,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,app_db,public,uuid_generate_v5_16396,app_db,public,uuid_generate_v5,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [69]:
# Создание процедуры на мастере 
result = master_conn.execute("""
DROP FUNCTION find_longest_string_in_table;
CREATE OR REPLACE FUNCTION find_longest_string_in_table(schema_name TEXT, table_name TEXT, column_name TEXT)
RETURNS TEXT AS $$
DECLARE
    longest_string TEXT;
BEGIN
    -- Построение SQL-запроса для поиска самой длинной строки
    EXECUTE format('SELECT %I FROM %I.%I ORDER BY LENGTH(%I) DESC LIMIT 1', 
        column_name, schema_name, table_name, column_name)
    INTO longest_string;
    
    -- Возврат найденной самой длинной строки
    RETURN longest_string;
END;
$$ LANGUAGE plpgsql STRICT;
""")
result

'Запрос выполнен успешно. Затронуто строк: -1'

In [66]:
# Получение списка процедур на реплике 
result = replica_conn.execute("""
select * 
from information_schema.routines
where routine_schema = 'public'
""")
result

,specific_catalog,specific_schema,specific_name,routine_catalog,routine_schema,routine_name,routine_type,module_catalog,module_schema,module_name,...,result_cast_interval_type,result_cast_interval_precision,result_cast_type_udt_catalog,result_cast_type_udt_schema,result_cast_type_udt_name,result_cast_scope_catalog,result_cast_scope_schema,result_cast_scope_name,result_cast_maximum_cardinality,result_cast_dtd_identifier
0,app_db,public,uuid_nil_16387,app_db,public,uuid_nil,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,app_db,public,uuid_ns_dns_16388,app_db,public,uuid_ns_dns,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,app_db,public,uuid_ns_url_16389,app_db,public,uuid_ns_url,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,app_db,public,uuid_ns_oid_16390,app_db,public,uuid_ns_oid,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,app_db,public,uuid_ns_x500_16391,app_db,public,uuid_ns_x500,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,app_db,public,uuid_generate_v1_16392,app_db,public,uuid_generate_v1,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,app_db,public,uuid_generate_v1mc_16393,app_db,public,uuid_generate_v1mc,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,app_db,public,uuid_generate_v3_16394,app_db,public,uuid_generate_v3,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
8,app_db,public,uuid_generate_v4_16395,app_db,public,uuid_generate_v4,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,app_db,public,uuid_generate_v5_16396,app_db,public,uuid_generate_v5,FUNCTION,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [72]:
# Запуск процедуры на реплике 
result = replica_conn.execute("""
SELECT * FROM find_longest_string_in_table('public', 'users', 'email');
""")
result

'customer1@example.com'

In [ ]:
# Закрытие соединения
replica_conn.close()
master_conn.close()